# Module 02 — Bitwise Operations as Algebra
### Mathematics of Cryptography · Python Notebook

---

This notebook is a companion to the HTML tutorial. Where the tutorial explains *why* bitwise operations matter, this notebook lets you **run the operations yourself** and see exactly what happens to the bits.

**What you will do in this notebook:**
- Represent bytes in binary, hex, and decimal side by side
- Run XOR, AND, OR, NOT, shifts, and rotations on real byte values
- Implement a working XOR cipher and decrypt a hidden message
- See where each operation appears inside real cryptographic algorithms

**How to use this notebook:**
1. Run each cell in order with **Shift + Enter**
2. Read the output before moving to the next cell
3. When you see `# YOUR TURN`, edit the cell and run it yourself
4. Solution cells are marked — try the exercise before peeking

> **No prior Python required.** Every line of code is explained as you go.

---
## Setup — Helper Functions

Run this cell first. It defines two display helpers used throughout the notebook so you can see every byte in **decimal**, **hex**, and **binary** at the same time.

In [ ]:
def show(value, label="value"):
    """Display a byte in decimal, hex, and binary."""
    v = value & 0xFF          # keep only the low 8 bits
    print(f"  {label:<14} dec={v:>3}   hex={v:#04x}   bin={v:08b}")

def show_op(a, b, result, op_symbol):
    """Display a two-operand bitwise operation with aligned columns."""
    a, b, r = a & 0xFF, b & 0xFF, result & 0xFF
    print(f"  {'A':<14} dec={a:>3}   hex={a:#04x}   bin={a:08b}")
    print(f"  {'B':<14} dec={b:>3}   hex={b:#04x}   bin={b:08b}")
    print(f"  {'─'*52}")
    print(f"  {('A '+op_symbol+' B'):<14} dec={r:>3}   hex={r:#04x}   bin={r:08b}")

print("Helper functions loaded. Ready to go.")

---
## Section 1 — Bytes and Representations

Before operating on bits we need to see them. Python can display any integer in binary (`bin()`), hexadecimal (`hex()`), or decimal. The number itself does not change — only the notation does.

The byte value **0x57** will follow us through this entire notebook. It is the ASCII code for the letter `W` and the same example used in the HTML tutorial.

In [ ]:
# Three ways to write the same number
a_decimal     = 87        # decimal literal
a_hex         = 0x57      # hex literal  (prefix 0x)
a_binary      = 0b01010111  # binary literal (prefix 0b)

print("All three refer to the same value:")
print(f"  decimal  87  == hex 0x57  ? {a_decimal == a_hex}")
print(f"  hex 0x57     == bin 0b01010111 ? {a_hex == a_binary}")
print()

# Displaying the same value in all three notations
print("Python built-in conversion functions:")
print(f"  bin(0x57) = {bin(0x57)}")
print(f"  hex(87)   = {hex(87)}")
print(f"  int('01010111', 2) = {int('01010111', 2)}")
print()

# Our helper gives a cleaner view
print("Using the show() helper:")
show(0x57, "byte 0x57")
show(0x83, "byte 0x83")

**What to notice:** `0x57` in binary is `01010111`. The two hex digits map directly to two 4-bit nibbles: `0101` (= 5) and `0111` (= 7). This is why cryptographers use hex — each digit is exactly four bits.

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# Use show() to display the byte 0xD4 in all three representations.
# Predict: how many 1-bits does 0xD4 have before running the cell?

show(0xD4, "byte 0xD4")   # run this line as-is, then try other values

---
## Section 2 — XOR: The Workhorse of Cryptography

XOR (exclusive OR) is the single most important bitwise operation in cryptography. Its rule is simple:

| A | B | A ⊕ B |
|---|---|-------|
| 0 | 0 |   0   |
| 0 | 1 |   1   |
| 1 | 0 |   1   |
| 1 | 1 |   0   |

Plain English: **output is 1 when inputs differ, 0 when they match.**

In Python, XOR is the `^` operator.

In [ ]:
# XOR truth table — generated by code
print("XOR truth table")
print("  A   B   A^B")
print("  ─── ─── ───")
for a in [0, 1]:
    for b in [0, 1]:
        print(f"  {a}   {b}   {a ^ b}")

In [ ]:
# XOR two bytes — the same example from the tutorial
P = 0x57   # plaintext byte
K = 0x83   # key byte

C = P ^ K  # XOR

print("Encrypting with XOR:")
show_op(P, K, C, "XOR")
print()
print(f"  Result C = {hex(C)}")

**The self-inverse property** is what makes XOR useful for encryption. XORing with the same key *twice* returns the original value. This means the encryption and decryption functions are identical — no separate decrypt algorithm needed.

In [ ]:
# XOR is its own inverse: (P ^ K) ^ K == P
P = 0x57
K = 0x83

C         = P ^ K         # encrypt
recovered = C ^ K         # decrypt (same operation!)

print("Encrypt then decrypt:")
show(P,         "Original P")
show(C,         "Ciphertext C")
show(recovered, "Recovered P")
print()
print(f"  P == recovered ? {P == recovered}")

In [ ]:
# Three algebraic properties of XOR — verify them all
x, y, z = 0x57, 0x83, 0x1F

print("Property 1 — Self-inverse:   x ^ x = 0")
print(f"  0x{x:02x} ^ 0x{x:02x} = {x ^ x}  ✓" if (x ^ x == 0) else "  FAIL")
print()
print("Property 2 — Identity:       x ^ 0 = x")
print(f"  0x{x:02x} ^ 0x00 = 0x{(x^0):02x}  ✓" if (x ^ 0 == x) else "  FAIL")
print()
print("Property 3 — Commutativity:  x ^ y = y ^ x")
print(f"  0x{x:02x} ^ 0x{y:02x} = {x^y}  ==  0x{y:02x} ^ 0x{x:02x} = {y^x}  ✓" if (x^y == y^x) else "  FAIL")
print()
print("Property 4 — Associativity:  (x ^ y) ^ z = x ^ (y ^ z)")
lhs = (x ^ y) ^ z
rhs = x ^ (y ^ z)
print(f"  LHS = {lhs},  RHS = {rhs}  ✓" if lhs == rhs else "  FAIL")

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# 1. XOR the bytes 0xA3 and 0x5C. What is the result in hex?
# 2. Verify the self-inverse property: XOR your result with 0x5C again.
#    Do you recover 0xA3?

result = 0xA3 ^ 0x5C          # change these values to experiment
show_op(0xA3, 0x5C, result, "XOR")
print()
print(f"  Self-inverse check: {hex(result)} ^ 0x5c = {hex(result ^ 0x5C)}")
print(f"  Recovered 0xA3? {result ^ 0x5C == 0xA3}")

---
## Section 3 — AND: Masking Bits

AND outputs 1 only when **both** inputs are 1. Its main job in cryptography is **masking** — isolating or extracting specific bits from a byte.

| A | B | A AND B |
|---|---|--------|
| 0 | 0 |    0   |
| 0 | 1 |    0   |
| 1 | 0 |    0   |
| 1 | 1 |    1   |

In Python, AND is the `&` operator.

In [ ]:
# Masking: extract the low nibble (lower 4 bits) of a byte
value = 0x57
mask  = 0x0F   # 00001111 — keeps only the low 4 bits

low_nibble = value & mask

print("Extracting the low nibble with AND:")
show_op(value, mask, low_nibble, "AND")
print()
print(f"  Low nibble of 0x57 = {hex(low_nibble)}  (decimal {low_nibble})")

In [ ]:
# Extract the HIGH nibble using a different mask
value     = 0x57
high_mask = 0xF0   # 11110000

high_nibble = (value & high_mask) >> 4   # shift right 4 to normalize

print("Extracting the high nibble:")
show_op(value, high_mask, value & high_mask, "AND")
print(f"  After right-shifting 4: {high_nibble}")
print(f"  High nibble of 0x57 = {high_nibble}  (the '5' in 0x57)")

**Cryptography connection:** The AES S-box lookup splits a byte into its high and low nibbles exactly this way. AND masking also appears in the **Ch** function inside SHA-256:

```
Ch(x, y, z) = (x AND y) XOR ((NOT x) AND z)
```

This acts as a bit-level multiplexer: where x = 1, take the bit from y; where x = 0, take the bit from z.

In [ ]:
# Implement the SHA-256 Ch function
def Ch(x, y, z):
    return (x & y) ^ (~x & z) & 0xFFFFFFFF

# Example with 8-bit values to see it clearly
x = 0b11001010
y = 0b10110011
z = 0b01010101

result = (x & y) ^ ((~x & 0xFF) & z)   # 8-bit version

print("SHA-256 Ch function (8-bit demo):")
show(x, "x (selector)")
show(y, "y (if x=1)")
show(z, "z (if x=0)")
print()
show(result, "Ch(x,y,z)")
print()
print("Bit by bit: where x has a 1, Ch takes from y; where x has a 0, Ch takes from z.")

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# Use AND to check whether the most significant bit (bit 7) of a byte is set.
# The mask for bit 7 is 0x80 (= 10000000 in binary).
#
# Test with: 0x57, 0x83, 0xD4
# Which of these have bit 7 set?

for val in [0x57, 0x83, 0xD4]:
    bit7 = (val & 0x80) != 0
    show(val, f"0x{val:02X}")
    print(f"    Bit 7 set? {bit7}")
    print()

---
## Section 4 — OR: Setting Bits

OR outputs 1 when **at least one** input is 1. Its job is **setting** specific bits to 1 without disturbing the others.

| A | B | A OR B |
|---|---|-------|
| 0 | 0 |   0   |
| 0 | 1 |   1   |
| 1 | 0 |   1   |
| 1 | 1 |   1   |

In Python, OR is the `|` operator.

In [ ]:
# OR to force specific bits ON
value   = 0x50   # 01010000
set_low = 0x07   # 00000111 — force the low 3 bits to 1

result = value | set_low

print("Using OR to set bits:")
show_op(value, set_low, result, "OR ")
print()
print("The low 3 bits are now all 1, the rest are unchanged.")

In [ ]:
# Practical use: combine two nibbles into one byte
high = 0x50   # high nibble already in position: 01010000
low  = 0x07   # low nibble:                       00000111

combined = high | low

print("Combining two nibbles with OR:")
show(high,     "high nibble")
show(low,      "low nibble")
show(combined, "combined")
print()
print(f"  0x{high:02X} | 0x{low:02X} = 0x{combined:02X}")

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# The high nibble of 0xA0 is 0xA (= 1010).
# The low nibble you want is 0x3  (= 0011).
# Use OR to assemble the byte 0xA3.

high = 0xA0
low  = 0x03

result = high | low
print(f"  Result: 0x{result:02X}")
print(f"  Correct? {result == 0xA3}")

---
## Section 5 — NOT: Flipping All Bits

NOT flips every bit. In an 8-bit byte, every 0 becomes 1 and every 1 becomes 0.

**Python gotcha:** Python's `~x` uses signed integers, so `~0x57` gives `-88`, not `0xA8`. For 8-bit NOT we mask with `0xFF`.

In [ ]:
value = 0x57

# Python's ~ uses two's complement (signed) — not what we want
print(f"  Python ~0x57        = {~0x57}  (signed, not useful here)")
print()

# 8-bit NOT: flip bits and keep only low 8
not_value = (~value) & 0xFF

print("8-bit NOT (mask with 0xFF):")
show(value,     "original")
show(not_value, "NOT result")
print()
print(f"  0x57 NOT = 0x{not_value:02X}")
print(f"  Verify: original AND NOT = {value & not_value} (should be 0)")
print(f"  Verify: original OR  NOT = {value | not_value} (should be 255 = 0xFF)")

In [ ]:
# Define a clean 8-bit NOT function
def NOT8(x):
    """8-bit NOT: flip all bits within one byte."""
    return (~x) & 0xFF

# ── YOUR TURN ─────────────────────────────────────────────────────────────
# Compute NOT8 of 0x83 and 0xD4.
# Notice: NOT8(NOT8(x)) should always return x.

for val in [0x57, 0x83, 0xD4]:
    n = NOT8(val)
    show(val, f"original 0x{val:02X}")
    show(n,   f"NOT8     0x{n:02X}")
    print(f"    Double NOT returns original? {NOT8(n) == val}")
    print()

---
## Section 6 — Bit Shifts: Multiplying and Dividing by Powers of 2

A **left shift** by n moves all bits n positions toward the high end. New zeros fill the right. Any bits shifted past position 7 are lost.

A **right shift** by n moves all bits n positions toward the low end. New zeros fill the left. Bits shifted past position 0 are lost.

**Key insight:** left shift by 1 is the same as multiplying by 2. Right shift by 1 is the same as integer division by 2.

In [ ]:
value = 0x0B   # 00001011 = 11 decimal

print("Left shifts (× 2 each step):")
print(f"  {'original':<16}  dec={value:>3}   bin={value:08b}")
for n in [1, 2, 3]:
    shifted = (value << n) & 0xFF
    print(f"  {'<< '+str(n):<16}  dec={shifted:>3}   bin={shifted:08b}   (= {value} × {2**n} = {value * 2**n} mod 256)")

In [ ]:
value = 0xB0   # 10110000 = 176 decimal

print("Right shifts (÷ 2 each step, integer division):")
print(f"  {'original':<16}  dec={value:>3}   bin={value:08b}")
for n in [1, 2, 3]:
    shifted = (value >> n) & 0xFF
    print(f"  {'>> '+str(n):<16}  dec={shifted:>3}   bin={shifted:08b}")

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# 1. Left-shift 0x57 by 1. What is the result in hex? What bits were lost?
# 2. Right-shift 0x57 by 2. What is the result?

v = 0x57
print("Original:")
show(v, "0x57")
print()

ls1 = (v << 1) & 0xFF
rs2 = (v >> 2) & 0xFF

print("Left shift by 1:")
show(ls1, "0x57 << 1")
print()

print("Right shift by 2:")
show(rs2, "0x57 >> 2")

---
## Section 7 — Rotations: Shifts That Wrap Around

A rotation is like a shift, but bits that fall off one end **reappear at the other end**. No information is lost.

Rotations appear everywhere in cryptography: AES's `RotWord`, SHA-256's `ROTR` and `ROTL` operations, and the Feistel round functions.

Python has no built-in rotation operator, so we implement it.

In [ ]:
def rotate_left(value, n, bits=8):
    """Rotate an integer left by n positions within a field of `bits` bits."""
    n = n % bits
    mask = (1 << bits) - 1
    return ((value << n) | (value >> (bits - n))) & mask

def rotate_right(value, n, bits=8):
    """Rotate an integer right by n positions within a field of `bits` bits."""
    n = n % bits
    mask = (1 << bits) - 1
    return ((value >> n) | (value << (bits - n))) & mask

value = 0x57   # 01010111

print("Rotate LEFT (bits wrap from left end to right end):")
print(f"  {'original':<16}  bin={value:08b}")
for n in [1, 2, 4]:
    r = rotate_left(value, n)
    print(f"  {'ROL '+str(n):<16}  bin={r:08b}   hex=0x{r:02X}")

print()
print("Rotate RIGHT (bits wrap from right end to left end):")
print(f"  {'original':<16}  bin={value:08b}")
for n in [1, 2, 4]:
    r = rotate_right(value, n)
    print(f"  {'ROR '+str(n):<16}  bin={r:08b}   hex=0x{r:02X}")

In [ ]:
# Rotations are reversible — rotate left then right returns original
original = 0x57
for n in [1, 3, 5]:
    rotated   = rotate_left(original, n)
    recovered = rotate_right(rotated, n)
    print(f"  ROL {n} then ROR {n}: 0x{original:02X} → 0x{rotated:02X} → 0x{recovered:02X}   recovered? {recovered == original}")

In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────
# SHA-256 uses 32-bit rotations. The ROTR operation in SHA-256 is:
#   ROTR^n(x) = rotate_right(x, n, bits=32)
#
# Try rotating 0x6A09E667 (the first SHA-256 initial hash value) right by 2.

sha_val = 0x6A09E667
rotated = rotate_right(sha_val, 2, bits=32)

print(f"  Original: {sha_val:#010x}   bin={sha_val:032b}")
print(f"  ROTR 2:   {rotated:#010x}   bin={rotated:032b}")

---
## Section 8 — XOR Encryption: Putting It All Together

A **stream cipher** encrypts data byte by byte. For each plaintext byte, XOR it with the corresponding key byte. The key is called a **keystream**.

This is how the one-time pad works, and it is also the basis of modern stream ciphers like ChaCha20 used in TLS 1.3.

The rule:
```
ciphertext[i] = plaintext[i] XOR keystream[i]
plaintext[i]  = ciphertext[i] XOR keystream[i]   ← same operation!
```

In [ ]:
def xor_bytes(data: bytes, key: bytes) -> bytes:
    """XOR each byte of data with the corresponding key byte (key repeats if shorter)."""
    return bytes(d ^ key[i % len(key)] for i, d in enumerate(data))

# Encrypt a message
plaintext  = b"CRYPTO"
key        = bytes([0x83, 0x1F, 0x4A, 0xC2, 0x77, 0xEB])

ciphertext = xor_bytes(plaintext, key)
decrypted  = xor_bytes(ciphertext, key)

print("XOR stream cipher demo:")
print(f"  Plaintext:   {plaintext}")
print(f"  Key:         {key.hex()}")
print(f"  Ciphertext:  {ciphertext.hex()}  ← looks like random bytes")
print(f"  Decrypted:   {decrypted}")
print()
print(f"  Decryption correct? {decrypted == plaintext}")

In [ ]:
# See it byte by byte
print("Step-by-step encryption:")
print(f"  {'Char':<6} {'P (dec)':<9} {'K (hex)':<9} {'C (hex)':<9} {'C (bin)'}")
print(f"  {'─'*56}")
for i, (p, k) in enumerate(zip(plaintext, key)):
    c = p ^ k
    print(f"  {chr(p):<6} {p:<9} {k:#04x}     {c:#04x}     {c:08b}")

In [ ]:
# ── CHALLENGE ─────────────────────────────────────────────────────────────
# A message was encrypted with the repeating key b"KEY".
# Decrypt it to find the hidden message.
#
# Hint: decryption is the same as encryption — XOR with the same key.

ciphertext_hex = "0b2516061d450e1e091a"
ciphertext = bytes.fromhex(ciphertext_hex)
key = b"KEY"

# YOUR CODE: decrypt the message
message = xor_bytes(ciphertext, key)

print(f"  Ciphertext (hex): {ciphertext_hex}")
print(f"  Key:              {key}")
print(f"  Decrypted:        {message.decode()}")

---
## Section 9 — Where These Operations Live in Real Cryptography

Every operation in this notebook appears in production cryptographic algorithms. This section connects what you built above to the actual code inside AES and SHA-256.

In [ ]:
# AES AddRoundKey — XOR the 16-byte state with a 16-byte round key
# This is exactly our xor_bytes function applied to a full 128-bit block

state     = bytes(range(16))              # 16 bytes: 00 01 02 ... 0F
round_key = bytes([0x2B]*8 + [0x7E]*8)   # simplified example key

after_ark = xor_bytes(state, round_key)

print("AES AddRoundKey (one round):")
print(f"  State:      {state.hex()}")
print(f"  Round key:  {round_key.hex()}")
print(f"  After XOR:  {after_ark.hex()}")
print()
print("Decryption recovers state:")
recovered = xor_bytes(after_ark, round_key)
print(f"  Recovered:  {recovered.hex()}")
print(f"  Correct?    {recovered == state}")

In [ ]:
# SHA-256 Σ0 function — built entirely from 32-bit rotations and XOR
# Σ0(x) = ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x)

def sigma0_sha256(x):
    rotr2  = rotate_right(x, 2,  bits=32)
    rotr13 = rotate_right(x, 13, bits=32)
    rotr22 = rotate_right(x, 22, bits=32)
    return rotr2 ^ rotr13 ^ rotr22

# Test with the first SHA-256 initial hash value (H0)
H0 = 0x6A09E667
result = sigma0_sha256(H0)

print("SHA-256 Σ0 function:")
print(f"  Input:  {H0:#010x}")
print(f"  ROTR 2: {rotate_right(H0, 2,  bits=32):#010x}")
print(f"  ROTR13: {rotate_right(H0, 13, bits=32):#010x}")
print(f"  ROTR22: {rotate_right(H0, 22, bits=32):#010x}")
print(f"  Σ0:     {result:#010x}")
print()
print("Notice: three rotations XOR'd together → 32 bits of diffusion from one input word.")

---
## Summary

| Operation | Python | Job in crypto |
|-----------|--------|---------------|
| XOR       | `^`    | Key mixing, stream ciphers, AddRoundKey |
| AND       | `&`    | Masking bits, Ch function (SHA-256) |
| OR        | `\|`   | Setting bits, combining nibbles |
| NOT       | `~x & 0xFF` | Bit inversion, Maj/Ch in SHA-256 |
| Left shift  | `<<` | Fast multiply by 2ⁿ, diffusion |
| Right shift | `>>` | Fast divide by 2ⁿ, extraction |
| Rotate left  | `rotate_left()`  | Diffusion without information loss |
| Rotate right | `rotate_right()` | Diffusion without information loss |

**The three-ingredient pattern** underlying AES and most block ciphers:
1. **Substitution** — S-box lookup (AND masking to split the nibbles)
2. **Permutation** — ShiftRows (rotation of byte positions)
3. **Key mixing** — AddRoundKey (XOR with round key)

Every one of those steps is built from operations you ran in this notebook.

In [ ]:
# ── FINAL CHALLENGE ────────────────────────────────────────────────────────
# Build a toy 1-round cipher that uses XOR, AND, and rotation together.
#
# Round function:
#   1. XOR the byte with the key
#   2. Rotate left by 3
#   3. AND with 0xAA (to scramble specific bits)
#   4. XOR with the key again
#
# Encrypt 0x57 with key 0x83. What is the ciphertext?
# Can you find the inverse operations to decrypt it?

def toy_encrypt(p, k):
    x = p ^ k
    x = rotate_left(x, 3)
    x = x & 0xAA
    x = x ^ k
    return x

P = 0x57
K = 0x83
C = toy_encrypt(P, K)

print("Toy cipher encryption:")
show(P, "Plaintext")
show(K, "Key")
show(C, "Ciphertext")
print()
print("Note: the AND step with 0xAA destroys information (bits at even positions")
print("become 0 regardless of input). This cipher is NOT reversible — can you see why?")
print("A real cipher must be a permutation: reversible for every key.")